In [1]:
import os
import json 
import time 
import re
import random
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

seed = 42
set_seed(seed)

/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. LFF v2

## Embedding Cosine similarity

In [9]:
base_output_path = "./output/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_test.jsonl"
lff2_output_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v2_test.jsonl"
lff2_similarity_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v2_test_sim.jsonl"

base_output = read_data(base_output_path)
lff2_output = read_data(lff2_output_path)
lff2_similarity = read_data(lff2_similarity_path)

print(f"base_output: {len(base_output)}")
print(f"lff2_output: {len(lff2_output)}")
print(f"lff2_similarity: {len(lff2_similarity)}")

base_output: 1319
lff2_output: 1319
lff2_similarity: 1319


In [10]:
all_similarity = []
correct_similarity = []
incorrect_similarity = []

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    pred_answer = base_output[i]["pred_ans"]
    similarity = lff2_similarity[i]["max_sim"]

    all_similarity.append(similarity)

    if true_answer == pred_answer:
        correct_similarity.append(similarity)
    else:
        incorrect_similarity.append(similarity)

print(f"all_similarity: {len(all_similarity)}")       
print(f"all_similarity: {np.mean(all_similarity)}")
print(f"correct_similarity: {len(correct_similarity)}")
print(f"correct_similarity: {np.mean(correct_similarity)}")
print(f"incorrect_similarity: {len(incorrect_similarity)}")
print(f"incorrect_similarity: {np.mean(incorrect_similarity)}")

100%|██████████| 1319/1319 [00:00<00:00, 799925.82it/s]

all_similarity: 1319
all_similarity: 0.38671504809514784
correct_similarity: 1074
correct_similarity: 0.38544667277700184
incorrect_similarity: 245
incorrect_similarity: 0.3922751913265306


## Correct - Incorrect

In [12]:
correct_correct = 0
correct_incorrect = 0
incorrect_correct = 0
incorrect_incorrect = 0

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    base_pred_answer = base_output[i]["pred_ans"]
    lff2_pred_answer = lff2_output[i]["pred_ans"]

    if base_pred_answer == true_answer:
        if lff2_pred_answer == true_answer:
            correct_correct += 1
        else:
            correct_incorrect += 1
    else:
        if lff2_pred_answer == true_answer:
            incorrect_correct += 1
        else:
            incorrect_incorrect += 1

print(f"correct_correct: {correct_correct}")
print(f"correct_incorrect: {correct_incorrect}")
print(f"incorrect_correct: {incorrect_correct}")
print(f"incorrect_incorrect: {incorrect_incorrect}")
print(f"total num: {correct_correct + correct_incorrect + incorrect_correct + incorrect_incorrect}")

100%|██████████| 1319/1319 [00:00<00:00, 630963.39it/s]

correct_correct: 939
correct_incorrect: 135
incorrect_correct: 84
incorrect_incorrect: 161
total num: 1319


# 2. LFF v3

## Embedding Cosine similarity

In [13]:
base_output_path = "./output/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_test.jsonl"
lff3_output_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v3_test.jsonl"
lff3_similarity_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v3_test_sim.jsonl"

base_output = read_data(base_output_path)
lff3_output = read_data(lff3_output_path)
lff3_similarity = read_data(lff3_similarity_path)

print(f"base_output: {len(base_output)}")
print(f"lff3_output: {len(lff3_output)}")
print(f"lff3_similarity: {len(lff3_similarity)}")

base_output: 1319
lff3_output: 1319
lff3_similarity: 1319


In [14]:
all_similarity = []
correct_similarity = []
incorrect_similarity = []

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    pred_answer = base_output[i]["pred_ans"]
    similarity = lff3_similarity[i]["max_sim"]

    all_similarity.append(similarity)

    if true_answer == pred_answer:
        correct_similarity.append(similarity)
    else:
        incorrect_similarity.append(similarity)

print(f"all_similarity: {len(all_similarity)}")       
print(f"all_similarity: {np.mean(all_similarity)}")
print(f"correct_similarity: {len(correct_similarity)}")
print(f"correct_similarity: {np.mean(correct_similarity)}")
print(f"incorrect_similarity: {len(incorrect_similarity)}")
print(f"incorrect_similarity: {np.mean(incorrect_similarity)}")

100%|██████████| 1319/1319 [00:00<00:00, 658213.80it/s]

all_similarity: 1319
all_similarity: 0.28476421827615617
correct_similarity: 1074
correct_similarity: 0.2844397404562384
incorrect_similarity: 245
incorrect_similarity: 0.2861866230867347


## Correct - Incorrect

In [15]:
correct_correct = 0
correct_incorrect = 0
incorrect_correct = 0
incorrect_incorrect = 0

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    base_pred_answer = base_output[i]["pred_ans"]
    lff3_pred_answer = lff3_output[i]["pred_ans"]

    if base_pred_answer == true_answer:
        if lff3_pred_answer == true_answer:
            correct_correct += 1
        else:
            correct_incorrect += 1
    else:
        if lff3_pred_answer == true_answer:
            incorrect_correct += 1
        else:
            incorrect_incorrect += 1

print(f"correct_correct: {correct_correct}")
print(f"correct_incorrect: {correct_incorrect}")
print(f"incorrect_correct: {incorrect_correct}")
print(f"incorrect_incorrect: {incorrect_incorrect}")
print(f"total num: {correct_correct + correct_incorrect + incorrect_correct + incorrect_incorrect}")

100%|██████████| 1319/1319 [00:00<00:00, 788974.18it/s]

correct_correct: 948
correct_incorrect: 126
incorrect_correct: 93
incorrect_incorrect: 152
total num: 1319
